<a href="https://colab.research.google.com/github/MawaSC/Quantum-Simulation-on-IQM-Resonance/blob/main/05_iqm_hardware.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
sys.path.append("../src")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qrisp.interface import IQMBackend

from fermi_hubbard import (
    L,
    init_occ,
    build_circuit,
    occupations_from_counts
)

1. Execute the circuit on IQM hardware

We run the selected Trotterized circuit on the configured IQM backend and collect the measurement results.

We record the circuit settings and number of shots along with the results so we can compare runs consistently. Since real hardware is affected by noise, measurement uncertainty, compilation, and execution conditions, its results may differ from both the ideal simulation and our simplified noise simulation.

The IQM API token is loaded from an environment variable rather than being stored in the notebook or GitHub repository.

In [ ]:
api_token = os.getenv("IQM_API_TOKEN")

if not api_token:
    raise ValueError(
        "IQM_API_TOKEN environment variable is not set."
    )

quantum_computer = IQMBackend(
    api_token=api_token,
    server_url="https://resonance.iqm.tech/",
    device_instance="garnet",
    use_timeslot=False
)

In [ ]:
# Choose experimental parameters
dt = 0.10

# Example: hardware run at t = 3
n_steps = int(3.0 / dt)

qv = build_circuit(
    dt=dt,
    n_steps=n_steps,
    init_occ=init_occ
)

counts_hardware = qv.get_measurement(
    backend=quantum_computer,
    shots=4000
)

# Convert measurements into occupations
nup_hardware, ndn_hardware = occupations_from_counts(
    counts_hardware,
    flipped=(n_steps % 2 == 1)
)

print("Spin-up occupations:", nup_hardware)
print("Spin-down occupations:", ndn_hardware)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Convert occupations to numpy arrays
nup_hardware = np.array(nup_hardware)
ndn_hardware = np.array(ndn_hardware)
total_hardware = nup_hardware + ndn_hardware

sites = np.arange(len(total_hardware))

# 2. Hardware Total Occupation Plot

plt.figure(figsize=(8, 5))

plt.plot(
    sites,
    nup_hardware,
    "o--",
    linewidth=2,
    label="Hardware Spin-Up",
)

plt.plot(
    sites,
    ndn_hardware,
    "s--",
    linewidth=2,
    label="Hardware Spin-Down",
)

plt.plot(
    sites,
    total_hardware,
    "^-",
    linewidth=2,
    label="Hardware Total Occupation",
)

plt.xlabel("Lattice Site")
plt.ylabel("Total Occupation")
plt.title("IQM Garnet Hardware Occupation (t = 3.0)")

plt.xticks(sites)
plt.legend()

plt.grid()

# Save as PNG and PDF (dpi=300 for crisp quality, bbox_inches='tight' prevents edge cropping)
plt.savefig("iqm_hardware_occupation_plot.png", bbox_inches="tight", dpi=300)
plt.savefig("iqm_hardware_occupation_plot.pdf", bbox_inches="tight", dpi=300)

plt.show()

# 3. Hardware Results Table

results_table = pd.DataFrame({
    "Lattice Site": sites,
    "Hardware Spin-Up": nup_hardware,
    "Hardware Spin-Down": ndn_hardware,
    "Hardware Total": total_hardware,
})

results_table = results_table.round({
    "Hardware Spin-Up": 3,
    "Hardware Spin-Down": 3,
    "Hardware Total": 3,
})

print(results_table.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. Calculate Total Occupation for Hardware
nup_hardware = np.array(nup_hardware)
ndn_hardware = np.array(ndn_hardware)
total_hardware = nup_hardware + ndn_hardware

sites = np.arange(len(total_hardware))


# 2. Hardware Total Occupation Plot

plt.figure(figsize=(8, 5))

plt.plot(
    sites,
    total_hardware,
    '^-',
    linewidth=2,
    label="IQM Garnet Hardware"
)

plt.xlabel("Lattice Site")
plt.ylabel("Total Occupation")
plt.title("IQM Garnet Hardware Occupation (t = 3.0)")

plt.xticks(sites)
plt.legend()

plt.grid()

# Save as PNG and PDF (dpi=300 for crisp quality, bbox_inches='tight' prevents edge cropping)
plt.savefig('hardware_total_occupation_plot.png', bbox_inches='tight', dpi=300)
plt.savefig('hardware_total_occupation_plot.pdf', bbox_inches='tight', dpi=300)

plt.show()

# 3. Hardware Results Table

results_table = pd.DataFrame({
    "Lattice Site": sites,
    "Hardware Total": total_hardware
})

results_table = results_table.round({
    "Hardware Total": 3
})

print(results_table.to_string(index=False))

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    sites,
    ideal_total,
    'o-',
    linewidth=2,
    label="Ideal Simulation"
)

plt.errorbar(
    sites,
    mean_total,
    yerr=std_total,
    fmt='s--',
    capsize=4,
    linewidth=2,
    label="Depolarizing Noise (mean ± SD)"
)

plt.plot(
    sites,
    total_hardware,
    '^-',
    linewidth=2,
    label="IQM Garnet Hardware"
)

plt.xlabel("Lattice Site")
plt.ylabel("Total Occupation")
plt.title("Total Occupation Comparison (t = 3.0)")

plt.xticks(sites)
plt.legend()

plt.grid()

plt.savefig('ideal_vs_noise_vs_hardware_plot.png', bbox_inches='tight', dpi=300)
plt.savefig('ideal_vs_noise_vs_hardware_plot.pdf', bbox_inches='tight', dpi=300)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# 1. Hardware Excution - 5 runs
target_time = 3.0
n_steps = int(target_time / dt)  # 30 steps
shots = 4096
num_runs = 5  # Replicating 5 independent hardware executions

hw_up_runs = []
hw_down_runs = []
hw_total_runs = []

print(f"Running circuit on IQM hardware across {num_runs} independent runs...")

for i in range(num_runs):
    print(f"  -> Executing hardware run {i+1}/{num_runs}...")

    # Build fresh circuit instance
    qv = build_circuit(dt=dt, n_steps=n_steps, init_occ=init_occ)

    # Measure on physical backend
    counts_hw = qv.get_measurement(backend=quantum_computer, shots=shots)

    # Extract occupations
    nup_hw, ndn_hw = occupations_from_counts(
        counts_hw, flipped=(n_steps % 2 == 1)
    )
    nup_hw = np.array(nup_hw)
    ndn_hw = np.array(ndn_hw)
    total_hw = nup_hw + ndn_hw

    # Store run arrays
    hw_up_runs.append(nup_hw)
    hw_down_runs.append(ndn_hw)
    hw_total_runs.append(total_hw)

# Convert lists of runs to 2D numpy arrays shape (num_runs, num_sites)
hw_up_runs = np.array(hw_up_runs)
hw_down_runs = np.array(hw_down_runs)
hw_total_runs = np.array(hw_total_runs)


# 2. Mean & SD across runs
mean_hw_total = np.mean(hw_total_runs, axis=0)
std_hw_total = np.std(hw_total_runs, axis=0, ddof=1)

sites = np.arange(len(mean_hw_total))

# 3. Errorbar

plt.figure(figsize=(8, 5))

plt.errorbar(
    sites,
    mean_hw_total,
    yerr=std_hw_total,
    fmt="^-",
    capsize=4,
    linewidth=2,
    label="IQM Garnet Hardware (mean ± SD)",
)

plt.xlabel("Lattice Site")
plt.ylabel("Total Occupation")
plt.title("IQM Garnet Hardware Occupation (t = 3.0, 5 Runs)")

plt.xticks(sites)
plt.legend()
plt.grid()

# Save figures
plt.savefig("hardware_mean_SD_plot.png", bbox_inches="tight", dpi=300)
plt.savefig("hardware_mean_SD_plot.pdf", bbox_inches="tight", dpi=300)

plt.show()


# 4. Summary
hw_results_table = pd.DataFrame({
    "Lattice Site": sites,
    "Hardware Mean": mean_hw_total,
    "Hardware SD": std_hw_total,
})

hw_results_table = hw_results_table.round(
    {"Hardware Mean": 3, "Hardware SD": 3}
)

print("\n--- IQM Garnet Hardware Statistical Results ---")
print(hw_results_table.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# 1. Plots Combined

plt.figure(figsize=(8, 5))

# 1. Ideal Simulation
plt.plot(
    sites,
    ideal_total,
    'o-',
    linewidth=2,
    label="Ideal Simulation"
)

# 2. Depolarizing Noise Model (mean ± SD)
plt.errorbar(
    sites,
    mean_total,
    yerr=std_total,
    fmt='s--',
    capsize=4,
    linewidth=2,
    label="Depolarizing Noise (mean ± SD)"
)

# 3. IQM Hardware Execution (mean ± SD)
plt.errorbar(
    sites,
    mean_hw_total,
    yerr=std_hw_total,
    fmt='^-',
    capsize=4,
    linewidth=2,
    label="IQM Hardware (mean ± SD)"
)

plt.xlabel("Lattice Site")
plt.ylabel("Total Occupation")
plt.title("Total Occupation Comparison (t = 3.0)")

plt.xticks(sites)
plt.legend()

plt.grid()

# Save as PNG and PDF (dpi=300 for crisp quality, bbox_inches='tight' prevents edge cropping)
plt.savefig('ideal_vs_noise_vs_hardware_plot.png', bbox_inches='tight', dpi=300)
plt.savefig('ideal_vs_noise_vs_hardware_plot.pdf', bbox_inches='tight', dpi=300)

plt.show()


# 2. Comparison Summary Table

comparison_table = pd.DataFrame({
    "Lattice Site": sites,
    "Ideal Occupation": ideal_total,
    "Noisy Mean": mean_total,
    "Noisy SD": std_total,
    "Hardware Mean": mean_hw_total,
    "Hardware SD": std_hw_total
})

comparison_table = comparison_table.round({
    "Ideal Occupation": 3,
    "Noisy Mean": 3,
    "Noisy SD": 3,
    "Hardware Mean": 3,
    "Hardware SD": 3
})

print("\n--- Benchmark Summary Table ---")
print(comparison_table.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


# 1. Results

results_table = pd.DataFrame({
    "Lattice Site": sites,
    "Ideal Occupation": ideal_total,
    "Noisy Mean": mean_total,
    "Noisy SD": std_total,
    "Hardware Mean": mean_hw_total,
    "Hardware SD": std_hw_total
})

# Round numeric values for clean table display
results_table = results_table.round({
    "Ideal Occupation": 3,
    "Noisy Mean": 3,
    "Noisy SD": 3,
    "Hardware Mean": 3,
    "Hardware SD": 3
})

fig, ax = plt.subplots(
    figsize=(9, len(results_table) * 0.5 + 1.2),
    dpi=300
)

ax.axis('off')

#Table created

table = ax.table(
    cellText=results_table.values,
    colLabels=results_table.columns,
    cellLoc='center',
    loc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)


header_color = '#1f4e78'
zebra_color = '#f2f6fa'

for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor('#d0d7de')

    if row == 0:
        # Header Styling
        cell.set_facecolor(header_color)
        cell.get_text().set_color('white')
        cell.get_text().set_weight('bold')
    else:
        # Alternating Row Colors (Zebra Striping)
        cell.set_facecolor(
            zebra_color if row % 2 == 0 else '#ffffff'
        )
        cell.get_text().set_color('#222222')

#Save

plt.savefig(
    'ideal_vs_noise_vs_hardware_results_table.png',
    bbox_inches='tight',
    dpi=300
)

plt.savefig(
    'ideal_vs_noise_vs_hardware_results_table.pdf',
    bbox_inches='tight',
    dpi=300
)

plt.close()

print("Comparison table successfully saved as PNG and PDF.")

In [ ]:
# Hardware time
hardware_time = 3.0

# Make sure hardware occupations are numpy arrays
nup_hardware = np.array(nup_hardware)
ndn_hardware = np.array(ndn_hardware)


fig, axes = plt.subplots(
    2, 2,
    figsize=(11, 6.5),
    sharex=True,
    sharey=True
)


for i, ax in enumerate(axes.flat):

    # -----------------------------
    # Simulation results
    # -----------------------------

    ax.plot(
        times,
        nup_q[:, i],
        'o-',
        ms=4,
        color='crimson',
        label=r'$\langle n_{\uparrow}\rangle$ Simulation'
    )

    ax.plot(
        times,
        ndn_q[:, i],
        's-',
        ms=4,
        color='royalblue',
        label=r'$\langle n_{\downarrow}\rangle$ Simulation'
    )


    # -----------------------------
    # Exact results
    # -----------------------------

    ax.plot(
        times,
        nup_e[:, i],
        '--',
        color='crimson',
        alpha=0.55,
        label='Exact'
    )

    ax.plot(
        times,
        ndn_e[:, i],
        '--',
        color='royalblue',
        alpha=0.55
    )


    # -----------------------------
    # Hardware results
    # -----------------------------

    ax.plot(
        hardware_time,
        nup_hardware[i],
        'o',
        ms=7,
        color='lightgreen',
        markeredgecolor='darkgreen',
        label=r'$\langle n_{\uparrow}\rangle$ Hardware'
    )

    ax.plot(
        hardware_time,
        ndn_hardware[i],
        's',
        ms=7,
        color='darkgreen',
        label=r'$\langle n_{\downarrow}\rangle$ Hardware'
    )


    # -----------------------------
    # Formatting
    # -----------------------------

    ax.set_title(f"Site {i}")
    ax.grid(alpha=0.3)
    ax.set_ylim(-0.05, 1.05)


# Legend
axes[0, 0].legend(fontsize=8, loc='center right')


# Axis labels
for ax in axes[1]:
    ax.set_xlabel(r"Time $t/t_h^{-1}$")

for ax in axes[:, 0]:
    ax.set_ylabel("Occupation")


# Overall title
fig.suptitle(
    f"1D Fermi-Hubbard Quench from a Néel State, "
    f"L={L}, U/t_h={U/t_h:.0f}"
)


fig.tight_layout()


# Save
plt.savefig(
    "fermi_hubbard_neel_quench_hardware.png",
    bbox_inches="tight",
    dpi=300
)

plt.savefig(
    "fermi_hubbard_neel_quench_hardware.pdf",
    bbox_inches="tight",
    dpi=300
)

plt.show()

In [ ]:
RUN_FULL_HARDWARE_HISTORY = False

In [ ]:
hardware_times = np.arange(0.0, 3.0 + dt, dt)

nup_hardware_history = []
ndn_hardware_history = []

for t in hardware_times:

    n_steps = int(round(t / dt))

    qv = build_circuit(
        dt=dt,
        n_steps=n_steps,
        init_occ=init_occ
    )

    counts_hardware = qv.get_measurement(
        backend=quantum_computer,
        shots=4000
    )

    nup_hardware, ndn_hardware = occupations_from_counts(
        counts_hardware,
        flipped=(n_steps % 2 == 1)
    )

    nup_hardware_history.append(nup_hardware)
    ndn_hardware_history.append(ndn_hardware)


nup_hardware_history = np.array(nup_hardware_history)
ndn_hardware_history = np.array(ndn_hardware_history)

In [ ]:
fig, axes = plt.subplots(
    2, 2,
    figsize=(11, 6.5),
    sharex=True,
    sharey=True
)

for i, ax in enumerate(axes.flat):

    # Simulation
    ax.plot(
        times,
        nup_q[:, i],
        'o-',
        ms=4,
        color='crimson',
        label=r'$\langle n_{\uparrow}\rangle$ Simulation'
    )

    ax.plot(
        times,
        ndn_q[:, i],
        's-',
        ms=4,
        color='royalblue',
        label=r'$\langle n_{\downarrow}\rangle$ Simulation'
    )

    # Exact
    ax.plot(
        times,
        nup_e[:, i],
        '--',
        color='crimson',
        alpha=0.55,
        label='Exact'
    )

    ax.plot(
        times,
        ndn_e[:, i],
        '--',
        color='royalblue',
        alpha=0.55
    )

    # Hardware
    ax.plot(
        hardware_times,
        nup_hardware_history[:, i],
        'o-',
        ms=4,
        linewidth=2,
        color='lightgreen',
        markeredgecolor='darkgreen',
        label=r'$\langle n_{\uparrow}\rangle$ Hardware'
    )

    ax.plot(
        hardware_times,
        ndn_hardware_history[:, i],
        's-',
        ms=4,
        linewidth=2,
        color='darkgreen',
        label=r'$\langle n_{\downarrow}\rangle$ Hardware'
    )

    ax.set_title(f"Site {i}")
    ax.grid(alpha=0.3)
    ax.set_ylim(-0.05, 1.05)

axes[0, 0].legend(fontsize=8, loc='lower right')
#axes[0, 0].legend(fontsize=8, loc='center right')

for ax in axes[1]:
    ax.set_xlabel(r"Time $t/t_h^{-1}$")

for ax in axes[:, 0]:
    ax.set_ylabel("Occupation")

fig.suptitle(
    f"1D Fermi-Hubbard Quench from a Néel State - Ideal vs. Hardware, "
    f"L={L}, U/t_h={U/t_h:.0f}",
    fontsize = 14.5
)

fig.tight_layout()

plt.savefig(
    "fermi_hubbard_neel_quench_hardware.png",
    bbox_inches="tight",
    dpi=300
)

plt.savefig(
    "fermi_hubbard_neel_quench_hardware.pdf",
    bbox_inches="tight",
    dpi=300
)

plt.show()

In [ ]:
fig, axes = plt.subplots(
    2, 2,
    figsize=(11, 6.5),
    sharex=True,
    sharey=True
)

for i, ax in enumerate(axes.flat):

    # Simulation
    ax.plot(
        times,
        nup_q[:, i],
        'o-',
        ms=4,
        color='crimson',
        label=r'$\langle n_{\uparrow}\rangle$ Simulation'
    )

    ax.plot(
        times,
        ndn_q[:, i],
        's-',
        ms=4,
        color='royalblue',
        label=r'$\langle n_{\downarrow}\rangle$ Simulation'
    )

    # Exact
    ax.plot(
        times,
        nup_e[:, i],
        '--',
        color='crimson',
        alpha=0.55,
        label='Exact'
    )

    ax.plot(
        times,
        ndn_e[:, i],
        '--',
        color='royalblue',
        alpha=0.55
    )


    ax.set_title(f"Site {i}")
    ax.grid(alpha=0.3)
    ax.set_ylim(-0.05, 1.05)

axes[0, 0].legend(fontsize=8, loc='lower right')
#axes[0, 0].legend(fontsize=8, loc='center right')

for ax in axes[1]:
    ax.set_xlabel(r"Time $t/t_h^{-1}$")

for ax in axes[:, 0]:
    ax.set_ylabel("Occupation")

fig.suptitle(
    f"1D Fermi-Hubbard Quench from a Néel State - Ideal vs. Hardware, "
    f"L={L}, U/t_h={U/t_h:.0f}",
    fontsize = 14.5
)

fig.tight_layout()

plt.savefig(
    "fermi_hubbard_neel_quench_hardware.png",
    bbox_inches="tight",
    dpi=300
)

plt.savefig(
    "fermi_hubbard_neel_quench_hardware.pdf",
    bbox_inches="tight",
    dpi=300
)

plt.show()

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ==============================================================================
# 1. PREPARE CIRCUIT DEPTH DATA (Simplified for t = 3.0 only)
# ==============================================================================
dt_values = [0.05, 0.10, 0.15, 0.20, 0.50]
summary = []

for dt in dt_values:
    n_max = int(3.0 / dt)
    qv_final = build_circuit(dt, n_max, init_occ)
    total_depth_final, two_qubit_depth_final = circuit_depths(qv_final)

    summary.append(
        {
            "dt": dt,
            "Target Time (t)": 3.0,
            "Trotter Steps (N)": n_max,
            "Total Circuit Depth": total_depth_final,
            "Two-Qubit Circuit Depth": two_qubit_depth_final,
        }
    )

depth_df = pd.DataFrame(summary)

# ==============================================================================
# 2. CREATE UNIFIED FIGURE LAYOUT (GridSpec)
# ==============================================================================
fig = plt.figure(figsize=(16, 11), dpi=300)
gs = gridspec.GridSpec(2, 2, height_ratios=[1.8, 1.0], hspace=0.35, wspace=0.28,
                       left=0.06, right=0.93, top=0.93, bottom=0.05)

# ------------------------------------------------------------------------------
# TOP LEFT: Néel State Quench Dynamics (Nested 2x2 Subgrid)
# ------------------------------------------------------------------------------
gs_top_left = gridspec.GridSpecFromSubplotSpec(
    2, 2, subplot_spec=gs[0, 0], hspace=0.32, wspace=0.25
)

axes_neel = [
    fig.add_subplot(gs_top_left[0, 0]),
    fig.add_subplot(gs_top_left[0, 1]),
    fig.add_subplot(gs_top_left[1, 0]),
    fig.add_subplot(gs_top_left[1, 1]),
]

for i, ax in enumerate(axes_neel):
    # Simulation
    ax.plot(
        times,
        nup_q[:, i],
        "o-",
        ms=3.5,
        color="crimson",
        label=r"$\langle n_{\uparrow}\rangle$ Sim",
    )
    ax.plot(
        times,
        ndn_q[:, i],
        "s-",
        ms=3.5,
        color="royalblue",
        label=r"$\langle n_{\downarrow}\rangle$ Sim",
    )

    # Exact
    ax.plot(
        times,
        nup_e[:, i],
        "--",
        color="crimson",
        alpha=0.55,
        label="Exact",
    )
    ax.plot(times, ndn_e[:, i], "--", color="royalblue", alpha=0.55)

    # Hardware
    ax.plot(
        hardware_times,
        nup_hardware_history[:, i],
        "o-",
        ms=3.5,
        linewidth=1.5,
        color="lightgreen",
        markeredgecolor="darkgreen",
        label=r"$\langle n_{\uparrow}\rangle$ HW",
    )
    ax.plot(
        hardware_times,
        ndn_hardware_history[:, i],
        "s-",
        ms=3.5,
        linewidth=1.5,
        color="darkgreen",
        label=r"$\langle n_{\downarrow}\rangle$ HW",
    )

    ax.set_title(f"Site {i}", fontsize=10, fontweight="bold")
    ax.grid(alpha=0.3)
    ax.set_ylim(-0.05, 1.05)

    # Set labels only on edges to keep grid clean
    if i in [2, 3]:
        ax.set_xlabel(r"Time $t/t_h^{-1}$", fontsize=9)
    if i in [0, 2]:
        ax.set_ylabel("Occupation", fontsize=9)

axes_neel[0].legend(fontsize=7, loc="lower right")

# ------------------------------------------------------------------------------
# TOP RIGHT: Total Occupation Comparison at t = 3.0
# ------------------------------------------------------------------------------
ax_comp = fig.add_subplot(gs[0, 1])

# Ideal
ax_comp.plot(
    sites, ideal_total, "o-", linewidth=2, markersize=6, label="Ideal Simulation"
)

# Depolarizing Noise
ax_comp.errorbar(
    sites,
    mean_total,
    yerr=std_total,
    fmt="s--",
    capsize=4,
    linewidth=2,
    markersize=6,
    label="Depolarizing Noise (mean ± SD)",
)

# IQM Hardware
ax_comp.errorbar(
    sites,
    mean_hw_total,
    yerr=std_hw_total,
    fmt="^-",
    capsize=4,
    linewidth=2,
    markersize=6,
    label="IQM Hardware (mean ± SD)",
)

ax_comp.set_xlabel("Lattice Site", fontsize=10, fontweight="bold")
ax_comp.set_ylabel("Total Occupation", fontsize=10, fontweight="bold")
ax_comp.set_title(
    "Total Occupation Comparison ($t = 3.0$)", fontsize=11, fontweight="bold"
)
ax_comp.set_xticks(sites)
ax_comp.grid(True, linestyle="--", alpha=0.5)
ax_comp.legend(frameon=True, facecolor="white", framealpha=0.9, fontsize=9)

# ------------------------------------------------------------------------------
# BOTTOM: Circuit Depth Scaling Table (Full Span)
# ------------------------------------------------------------------------------
ax_table = fig.add_subplot(gs[1, :])
ax_table.axis("off")

table_data = ax_table.table(
    cellText=depth_df.values,
    colLabels=depth_df.columns,
    cellLoc="center",
    loc="center",
    colWidths=[0.12, 0.15, 0.18, 0.22, 0.25]
)

table_data.auto_set_font_size(False)
table_data.set_fontsize(9.5)
table_data.scale(0.9, 1.8)

# Apply custom formatting
header_color = "#1f4e78"
zebra_color = "#f2f6fa"

for (row, col), cell in table_data.get_celld().items():
    cell.set_edgecolor("#d0d7de")

    if row == 0:
        cell.set_facecolor(header_color)
        cell.get_text().set_color("white")
        cell.get_text().set_weight("bold")
    else:
        cell.set_facecolor(zebra_color if row % 2 == 0 else "#ffffff")
        cell.get_text().set_color("#222222")

ax_table.set_title(
    "Circuit Depth Scaling vs. Trotter Step Size ($\Delta t$) at Target Time $t = 3.0$",
    fontsize=11,
    fontweight="bold",
    pad=12,
)

# ==============================================================================
# 3. SAVE AND SHOW
# ==============================================================================
plt.savefig("fermi_hubbard_unified_summary_page.png", bbox_inches="tight", dpi=300)
plt.savefig("fermi_hubbard_unified_summary_page.pdf", bbox_inches="tight", dpi=300)

plt.show()